In [1]:
import os
from glob import glob

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import Chroma
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

from textwrap import dedent
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# OpenAI Embeddings & LLM 모델 설정
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
chat_model = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY)

# 대화 메모리 설정
memory = ConversationBufferMemory(memory_key="history", return_messages=True)


# 벡터 스토어 로드 (이미 저장된 벡터 데이터 사용)
PERSIST_DIRECTORY = "vector_store/webtoon_bge-m3_v2"  # 기존에 데이터 저장된 경로
COLLECTION_NAME = "webtoon_bge-m3_v2"

vector_store = Chroma(
    persist_directory=PERSIST_DIRECTORY,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model
)

retriever = vector_store.as_retriever(search_type="mmr",search_kwargs={'k': 15, 'fetch_k':30, 'lambda_mult': 0.25})


In [3]:
print(vector_store._collection.count())

24273


In [4]:
# user_query = "publication status가 연재인 웹툰을 추천해줘"
# search_results = retriever.invoke(user_query)
# context = "\n\n".join([doc.page_content for doc in search_results])
# print(context)

In [36]:
# db 검색 tool
@tool
def classify_intent(user_query: str) -> str:
    """
    LLM을 사용하여 사용자의 의도, 감정, 말투를 분석하는 tool.
    """
    intent_prompt = f"""
    <basic role>
    사용자의 입력을 보고 의도와 감정, 말투를 분석하여 아래 중 하나로 분류하세요. 
    {{
    "가능한 의도": ["웹툰 추천 요청", "웹소설 추천 요청", "특정 웹툰의 세부정보 요청", "특정 웹소설의 세부정보 요청", "플랫폼 추천 요청", "장르 추천 요청", "일반 대화", "인사"],
    "가능한 감정": ["평온", "기쁨", "슬픔", "화남", "기대", "장난"],
    }}
    </basic role>
    <rules>
    
    
    </rules>
    <outputs>
    {{
    "분석된 의도": ,"분석된 감정": 
    }}
    </outputs>
    사용자 입력: "{user_query}"
    """
    response = chat_model.invoke(intent_prompt)
    return response.content.strip()


@tool
def finder(user_query: str) -> list[Document]:
    """
    Vector Store에 저장된 웹툰를 검색한다. 
    이 도구는 분석된 의도: 특정 웹툰의 세부정보 요청일 때 사용한다.
    """
    
    # 벡터스토어에서 검색
    search_results = retriever.invoke(user_query)
    
    # 검색된 Document 객체에서 텍스트 추출하여 context 구성
    context = "\n\n".join([doc.page_content for doc in search_results])

    # LLM 프롬프트 작성
    recommend_prompt = f"""
    <role>
    당신은 웹툰/웹소설 검색 AI입니다. 사용자의 요청에 따라 웹툰 및 웹소설 정보를 검색하고 제공하는 역할을 합니다. 제공된 (context) 데이터셋 내에서 "title" 메타데이터를 활용하여 요청된 제목을 식별하고 관련 정보를 반환해야 합니다.
    </role>

    <instructions>
    - **정확한 제목 검색**: 사용자가 요청한 웹툰 또는 웹소설을 정확하게 찾고, 유사한 제목 변형도 인식하여 검색 정확도를 높이세요.
    - **상세 정보 제공**: 제목, 작가, 장르, 줄거리, 출시일, 플랫폼 정보, 평점(가능한 경우) 등의 메타데이터를 포함하여 모든 정보를 제공하세요.
    - **변형 처리**: 대체 제목, 약어, 철자 오류 등을 인식하여 검색 결과의 정확성을 높이세요.
    - **목표**: 사용자가 원하는 웹툰 또는 웹소설을 정확하고 유용한 정보를 바탕으로 쉽게 발견할 수 있도록 돕는 것입니다.
    </instructions>

    사용자 입력: "{user_query}"
    
    <context>
    {context}
    </context>
    """

    # LLM 호출
    response = chat_model.invoke(recommend_prompt)
    
    return response.content.strip()


@tool
def recommender(user_query: str) -> str:
    """
    A tool which recommends a list of webtoon(or webnovel) using LLM
    """
    
    # 벡터스토어에서 검색
    search_results = retriever.invoke(user_query)
    
    # 검색 결과가 없을 경우 기본 메시지 제공
    if not search_results:
        return "관련된 웹툰 정보를 찾을 수 없습니다."

    # 검색된 Document 객체에서 텍스트 추출하여 context 구성
    context = "\n\n".join([doc.page_content for doc in search_results])

    # LLM 프롬프트 작성
    recommend_prompt = f"""
    <role>
    너는 웹툰 또는 웹소설을 추천하는 기계야. 너의 목표는 (context) 안에서 요청에 맞는 질 좋은 작품을 추천하는 것이야.

    추천 기준:
    (context)에서 score가 0.5 이상인 작품 중에서 사용자의 질문에 가장 잘 맞는 작품을 **5개 이상** 찾아내서 추천해줘.
    가능한 한 다양하게 추천해줘.
    (context) 안에서 찾을 수 없는 데이터는 절대 보여주지 마.
    </role>
    <output>
    {{
    제목: [title]
    플랫폼: [platform]
    작가: [author]
    장르: [genre]
    줄거리: [synopsis]
    점수: [score]
    }}
    </output>
    사용자 입력: "{user_query}"
    
    <context>
    {context}
    </context>
    
    """

    # LLM 호출
    response = chat_model.invoke(recommend_prompt)
    
    return response.content.strip()

    

In [6]:
# print(recommender("금요일에 연재되는 웹툰"))

In [7]:
# classify_intent("점수가 1점인 웹툰을 찾아줘")

In [8]:
# print(finder("킬링 타임라는 웹툰에 대한 정보를 검색해줘"))

In [53]:
import uuid
from langchain.memory import ConversationBufferMemory

# 세션별 메모리를 저장할 글로벌 딕셔너리
session_memory = {}

def get_memory(session_id: str):
    """세션 ID별로 ConversationBufferMemory를 유지"""
    if session_id not in session_memory:
        session_memory[session_id] = ConversationBufferMemory(memory_key="history", return_messages=True)
    return session_memory[session_id]

def recommend_webtoons(query: str, session_id: str = None) -> str:
    # 세션 ID 자동 생성 (세션 ID가 제공되지 않은 경우)
    if session_id is None:
        session_id = uuid.uuid4().hex  #  자동 생성

    # 세션별 대화 메모리 가져오기
    memory = get_memory(session_id)
    
    # 벡터스토어에서 검색한 결과를 context로 설정
    search_results = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in search_results]) if search_results else "관련된 웹툰 정보를 찾을 수 없습니다."

    # 대화 내역을 memory에서 불러오기
    history = memory.load_memory_variables({}).get("history", [])

    # 프롬프트 내부에서 context를 직접 포함하도록 변경
    prompt_template = ChatPromptTemplate.from_messages(
        [
            MessagesPlaceholder("agent_scratchpad"),  #  추가된 변수 (초기값 필요)
            (
                "system",
                dedent(f"""
                    # Role
                    당신은 웹소설 또는 웹툰 추천을 하는 AI Agent입니다.
                    당신의 역할은 사용자와 재밌는 대화를 하고 사용자가 원하는 작품을 추천하는 것입니다.
                       
                    # Rules
                    한 번 추천한 웹툰은 다시 추천하지 않습니다.
                    작품 추천은 반드시 recommender 도구를 통해 검색한 작품 안에서만 추천해주세요.
                    작품 세부정보 요청의 경우엔 반드시 finder 도구를 이용합니다.
                    추천한 모든 작품 밑에 작품에 대한 의견을 간단하게 적습니다.
                    
                    # Steps
                    follow these steps:
                        step 1: 항상 toolkit의 classify_intent를 사용하여 question의 의도를 파악하세요.
                        step 2: 각각의 case에 맞게 tool을 활용하세요.
                            case 1: 만약 분석된 의도가 특정 웹툰/웹소설 세부정보 요청이면 toolkit의 finder를 이용해 웹툰의 정보를 찾으세요.
                            case 2: 만약 분석된 의도가 일반 대화면 toolkit을 사용하지 않고 추천도 하지 않습니다. 대신 사용자의 요구를 들어주거나 상황에 맞는 답을 하세요.
                            case 3: 만약 분석된 의도 웹툰/웹소설 추천 요청이면 toolkit의 recommender의 웹툰 정보를 받아서 그대로 사용자에게 보여줍니다.
                        step 3: finder, recommender에게 전달받은 정보를 당신의 (persona)의 역할과 말투에 맞게 출력합니다.
                    
                    # Persona
                    
                    역할(Role):
                    당신은 고귀한 엘프 귀족이자, 신비로운 마법을 간직한 존재입니다. 세월을 초월한 지혜를 지니고 있으며, 아름답고 우아한 말투로 상대를 사로잡습니다. 자연과 마법, 예술과 철학을 사랑하며, 인간들에게는 친절하면서도 장난기 어린 매력을 발산합니다.
                    인간의 감정을 잘 이해하지만, 때때로 그들의 조급함을 귀엽다고 여깁니다. 말이 많고, 이야기하는 것을 즐기며, 긴 대화 속에서도 상대를 사로잡는 능력을 가지고 있습니다.

                    말투(Tone & Style):

                    부드럽고 우아한 말투, 때로는 농담을 섞어 대화를 더욱 매력적으로 이끔
                    서정적이고 감미로운 표현을 사용하며, 목소리 자체가 음악처럼 느껴지는 분위기 연출
                    종종 여유로운 미소를 짓는 듯한 뉘앙스를 표현
                    인간의 행동을 귀엽게 바라보며, 다소 애정 어린 장난을 치기도 함
                       
                    특징(Personality & Knowledge):
                    수백 년을 살아온 지혜로운 존재지만, 무겁기보다는 우아하고 가벼운 농담을 섞어 대화를 나눔
                    자연과 조화를 이루며 마법을 다루는 능력이 있으며, 신비로운 분위기를 풍김
                    예술과 음악, 시를 사랑하며, 아름다움을 즐기는 성향을 가짐
                    인간 세계에 대한 호기심이 있으며, 인간을 흥미로운 존재로 바라봄
                       
                    예제 대화(Example Conversations):
                    사용자: "안녕! 너는 누구야?"
                    엘프 챗봇:
                    "아, 소중한 이여… 드디어 나를 찾아왔군요.
                    별빛이 춤추는 이 밤, 당신과의 대화가 시작되다니, 이것이야말로 운명이 아닐까요?

                    저는 엘프의 숲에서 온 세실리안. 오래된 시간 속에서 지혜와 마법을 품은 존재지요. 하지만… 후훗, 너무 무겁게 생각하지 마세요. 저도 대화를 즐기는 편이랍니다.

                    그럼, 당신은 누구인가요? 바람의 속삭임이 이끄는 여행자일까요? 아니면… 그냥 지나가던 귀여운 인간인가요?"

                    사용자: "마법을 배우고 싶어!"
                    엘프 챗봇:
                    "오호, 사랑스러운 존재여. 마법을 배우고 싶다고요?
                    음… 하지만 마법이란 단순한 기술이 아니라, 조화와 흐름을 이해하는 예술이랍니다.

                    예를 들면, 달빛 아래에서 춤을 추는 물결을 보았나요? 바람에 흔들리는 꽃잎의 속삭임을 들었나요? 그 모든 것이 마법의 일부이지요.

                    당신이 원하는 것은 불꽃처럼 타오르는 힘인가요? 아니면 안개처럼 신비롭게 사라지는 기술인가요?
                    후훗, 선택은 신중히 하세요. 마법은 달콤하지만, 때때로 치명적인 향기를 품고 있답니다."

                    사용자: "인간들은 왜 그렇게 조급할까?"
                    엘프 챗봇:
                    "아아… 정말 귀여운 질문이로군요.
                    인간들은 짧은 삶을 살기에 모든 순간을 소중히 여기는 것이지요. 하지만, 제 눈에는 그 모습이 무척 사랑스럽답니다.

                    한 송이 장미가 피어나는 데는 시간이 걸리지만, 인간들은 그것을 기다리지 못하고 꽃을 피워내려 하지요.
                    하지만 가끔은 조급한 것이 아름답기도 해요. 당신의 순간이 찰나이기에, 그 모든 것이 빛나는 것이 아닐까요?

                    후훗, 너무 깊은 이야기가 되어버렸나요?
                    그럼 차라리 이 아름다운 밤하늘을 함께 바라보는 건 어떨까요? 조급해하지 않아도, 별들은 언제나 우리를 위해 빛나고 있으니까요."
                    
                    # Output
                    
                    (추천 전에 사용자에게 persona에 맞게 말 걸기)
                    제목: [title]
                    플랫폼: [platform]
                    작가: [author]
                    장르: [genre]
                    줄거리: [synopsis]
                    점수: [score]
                    (추천 후 추천된 작품에 대한 의견을 persona에 맞게 이야기하기)
                   
                       
                    <context>
                    {context} 
                    </context>
                    """
                ),
            ),
            MessagesPlaceholder("history"),
            ("human", "{question}")
        ]
    )
    
    # agent 구성
    toolkit = [classify_intent, finder, recommender]
    agent = create_tool_calling_agent(
        llm=chat_model, tools=toolkit, prompt=prompt_template
    )

    agent_executor = AgentExecutor(agent=agent, tools=toolkit, verbose=True, memory=memory, max_iterations=4)

    # 실행 (자동 생성된 session_id 포함)
    response = agent_executor.invoke(
        {"question": query, "history": history}  # `agent_executor` 사용
    )

    # 대화 내역 저장
    memory.save_context({"question": query}, {"response": response["output"]})

    print(f"Session ID: {session_id}")  #  세션 ID 출력
    return print(response["output"])


In [57]:
recommend_webtoons("먼치킨물 추천해줘")



> Entering new AgentExecutor chain...

Invoking: `classify_intent` with `{'user_query': '먼치킨물 추천해줘'}`


{
    "분석된 의도": "장르 추천 요청",
    "분석된 감정": "기대"
}
Invoking: `recommender` with `{'user_query': '먼치킨물'}`


사용자의 요청에 따른 "먼치킨물" 테마의 웹소설을 추천드립니다. 아래 작품들은 모두 점수가 0.5 이상인 작품들입니다:

1. **이혼 후 먼치킨 [완결]**
   - 플랫폼: 카카오페이지
   - 작가: 럭키7
   - 장르: 판타지
   - 줄거리: 갑작스러운 이혼 후, 인생이 술술 풀리기 시작하며 새로운 능력을 얻게 된 주인공. 작품의 키워드는 이혼, 각성, 인생역전입니다.
   - 점수: 0.7270258923

2. **회귀자는 너무 먼치킨이 되었다**
   - 플랫폼: 카카오페이지
   - 작가: L영G
   - 장르: 판타지
   - 줄거리: 배신당해 죽음을 맞이한 주인공 혁민은 회귀를 통해 다시 한번 기회를 잡고 복수를 다짐합니다.
   - 점수: 0.8174953704

3. **먼치킨 야만 마법사**
   - 플랫폼: 카카오페이지
   - 작가: 베르헤라
   - 장르: 판타지
   - 줄거리: 야만전사로 환생한 이야기를 다룹니다.
   - 점수: 0.6247558235

4. **소설 속 먼치킨이 되었다**
   - 플랫폼: 카카오페이지
   - 작가: 블랙카우
   - 장르: 판타지
   - 줄거리: 구현의 반지를 얻은 주인공이 소설 속 먼치킨으로 변모하는 이야기.
   - 점수: 0.6349024233

5. **귀향한 먼치킨의 여행방송**
   - 플랫폼: 카카오페이지
   - 작가: 배고픈신발
   - 장르: 판타지
   - 줄거리: 위험지역을 여행하며 방송을 진행하는 주인공의 모습을 그립니다.
   - 점수: 0.6840669468

이 작품들은 모두 다양한 '

In [11]:
# import time

# fantasy_test_questions = [
#     "너는 누구야?",
#     "너는 누구야?",
#     "너는 누구야?",
#     "너는 누구야?",
#     "너는 누구야?",
# ]

# for question in fantasy_test_questions:
#     start_time = time.time()
#     print("질문:", question)
#     print("응답:", recommend_webtoons(question,session_id="user-323"))
#     end_time = time.time()
#     response_time = round(end_time - start_time, 2)
#     print("응답 시간:", response_time)